# Tool 0: one participant, the whole pipeline

Explore **one recording** through every step of the Dream-Toolkit pipeline, with all the results shown here.
Use it to **diagnose a participant** (why was this night excluded? what does this artefact look like?) and to
**learn the pipeline** before running the batch tools on a whole database.

**How to use it**
1. Pick the **study folder** (the one holding `config_param/`, `derivatives/` and the `reports_*` folders; the
   recordings are found in any sub-folder), then the **participant**.
2. Go through the steps in order. Each tab says what the step does, which batch tool runs it on the whole
   database, and what to look at. Press its **Run** button.
3. The **epoch explorer** below the steps shows the night epoch by epoch as soon as the recording is loaded
   (step 1). You can rescore an epoch or add an annotation there.

**What it writes** (nothing else is ever modified):
- `<hypnogram>_rescored.txt` + `<hypnogram>_rescore_log.tsv` beside the hypnogram, when you save a rescoring;
- `<recording>_manual_events.tsv` beside the recording, when you add an annotation (the same file as tool 8's;
  tool 7 can include it in its event flagging);
- an inspection log and report in `reports_live_explore/` (untick *Keep an inspection report* to write nothing
  there).

Steps 3 to 8 are being built (tool-0 redesign): their tabs say so.

In [ ]:
try:
    import os
    import sys
    import io
    import json
    import base64
    import warnings
    from datetime import datetime
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patheffects as mpe
    import mne
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    from ipyfilechooser import FileChooser
    from scipy.signal import spectrogram as scipy_spectrogram

    # Shared libraries (tools/): found whether Voila is launched from the repo root or from tools/.
    _here = os.getcwd()
    for _cand in [_here, os.path.join(_here, 'tools'), os.path.join(os.path.dirname(_here), 'tools'),
                  os.path.join(_here, '..', 'tools')]:
        if os.path.isfile(os.path.join(_cand, 'edf_io_lib.py')):
            _cand = os.path.abspath(_cand)
            if _cand not in sys.path:
                sys.path.insert(0, _cand)
            break
    import edf_io_lib as EIO                    # EDF reading, the same code as the batch tools
    import qc_rejected_epochs_lib as L          # tool 8's display (montage, detail panel, annotations)
    import participant_selection_lib as PSL     # exclusion registry
except ImportError as e:
    print('⚠️ Import error, a package is missing:', e)
else:
    print('✅ Packages imported successfully!')

# ipyevents gives keyboard navigation in the explorer; optional (the buttons do everything without it).
try:
    from ipyevents import Event as _IpyEvent
    _HAS_IPYEVENTS = True
except Exception:
    _HAS_IPYEVENTS = False

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

In [ ]:
# ----- Steps, their status, and what a change makes out of date -----
SECTIONS = [('1', 'Load'), ('2', 'Header'), ('3', 'Scoring & events'), ('4', 'Macrostructure'),
            ('5', 'Quality'), ('6', 'Preprocessing'), ('7', 'Decision'), ('8', 'Spectral')]
# A rescoring or an annotation changes what these steps are computed from (stage durations, per-stage
# amplitude thresholds, event flags): they are marked OUT OF DATE, never recomputed behind the user's back.
AFTER_RESCORING = ['4', '6', '7', '8']
AFTER_ANNOTATION = ['4', '6', '7', '8']
DOWNSTREAM = {'1': ['3', '4', '5', '6', '7', '8'], '6': ['7', '8'], '7': ['8']}

STATE = {}                                  # the loaded participant (cleared on every participant change)
STATUS = {k: 'not_run' for k, _ in SECTIONS}
STATUS_WHY = {k: '' for k, _ in SECTIONS}
REPORT_BLOCKS = {}                          # step key -> HTML fragments shown in the inspection report
BADGES = {}                                 # step key -> its status widget (filled when the tabs are built)
TABS = None                                 # the Tab widget (its titles carry the status mark)


def badge_html(key):
    s = STATUS[key]
    if s == 'up_to_date':
        return ('<span style="background:#e8f5e9;color:#1b5e20;padding:2px 8px;border-radius:3px;">'
                'up to date</span>')
    if s == 'out_of_date':
        why = f': {STATUS_WHY[key]}' if STATUS_WHY[key] else ''
        return ('<span style="background:#fff3e0;color:#e65100;padding:2px 8px;border-radius:3px;">'
                f'<b>OUT OF DATE</b>{why}. Run it again.</span>')
    return '<span style="background:#f5f5f5;color:#757575;padding:2px 8px;border-radius:3px;">not run</span>'


def refresh_badges():
    for k, w in BADGES.items():
        w.value = badge_html(k)
    if TABS is not None:
        mark = {'up_to_date': ' ✓', 'out_of_date': ' ⚠', 'not_run': ''}
        for i, (k, name) in enumerate(SECTIONS):
            TABS.set_title(i, f'{k} {name}{mark[STATUS[k]]}')


def mark_out_of_date(keys, why):
    # Only a step that HAS results can become out of date; a step never run stays 'not run'.
    for k in keys:
        if STATUS[k] == 'up_to_date':
            STATUS[k] = 'out_of_date'
            STATUS_WHY[k] = why
    refresh_badges()


def mark_up_to_date(key):
    STATUS[key] = 'up_to_date'
    STATUS_WHY[key] = ''
    mark_out_of_date(DOWNSTREAM.get(key, []), f'step {key} was run again')


def unsaved_rescoring():
    """Number of epochs rescored since the last save (0 when nothing is loaded)."""
    if 'hypno_work' not in STATE:
        return 0
    return int((STATE['hypno_work'] != STATE['hypno_saved']).sum())


def guard_unsaved(what):
    """True when `what` may proceed. Unsaved rescoring blocks the FIRST attempt (warning); the same action
    repeated discards it, like tool 8's unsaved-changes guard."""
    n = unsaved_rescoring()
    if n == 0 or STATE.get('discard_armed') == what:
        return True
    STATE['discard_armed'] = what
    html_guard.value = (f'<p style="background:#fff3e0;color:#e65100;padding:4px 8px;"><b>⚠ {n} rescored '
                        f'epoch(s) not saved.</b> Save them in the explorer, or {what} again to discard them.</p>')
    return False


def reset_state():
    STATE.clear()
    for k in STATUS:
        STATUS[k] = 'not_run'
        STATUS_WHY[k] = ''
    REPORT_BLOCKS.clear()
    refresh_badges()


# ----- Figures -----
def fig_to_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    return buf.getvalue()


def png_html(png, alt=''):
    return (f'<img alt="{alt}" style="max-width:100%;" '
            f'src="data:image/png;base64,{base64.b64encode(png).decode()}">')


def training_box(what, batch, look):
    """The 'About this step' box at the top of each tab (training aid for a new user)."""
    return widgets.HTML(
        '<details open style="background:#f4f8fb;border-left:4px solid #2a6099;padding:6px 10px;'
        'margin:4px 0 8px 0;"><summary><b>About this step</b></summary>'
        f'<p style="margin:4px 0;"><b>What it does.</b> {what}</p>'
        f'<p style="margin:4px 0;"><b>On the whole database.</b> {batch}</p>'
        f'<p style="margin:4px 0;"><b>What to look at.</b> {look}</p></details>')


def df_html(df):
    return df.to_html(index=False, border=0, na_rep='', escape=True,
                      classes='dtk').replace('<table', '<table style="border-collapse:collapse;font-size:0.9em;"')


# ----- Inspection log + report (reports_live_explore/) -----
LOG_COLUMNS = ['file_id', 'time', 'step', 'action', 'details']


def report_folder():
    """<study>/reports_live_explore/<sub-folder of the recording>/ (mirrors the recordings' tree)."""
    study, edf = Path(STATE['study']), Path(STATE['edf'])
    try:
        sub = edf.parent.relative_to(study)
    except ValueError:
        sub = Path('.')
    return study / 'reports_live_explore' / sub


def log_action(step, action, details=''):
    """One action of the inspection: appended to {file_id}_inspection_log.tsv (kept across sessions, data
    first), then the report is rewritten. Non-fatal: a failure is shown, never raised."""
    if not cb_report.value or 'edf' not in STATE:
        return
    try:
        folder = report_folder()
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / f"{STATE['fid']}_inspection_log.tsv"
        row = pd.DataFrame([{'file_id': STATE['fid'], 'time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                             'step': step, 'action': action, 'details': details}], columns=LOG_COLUMNS)
        row.to_csv(path, sep='\t', index=False, mode='a', header=not path.exists())
    except Exception as e:
        html_report_info.value = f'<span style="color:#c62828;">⚠ inspection log not written: {e}</span>'
        return
    write_report()


def write_report():
    """Rewrite {file_id}_inspection_report.html: participant, step statuses, the whole log, and each
    step's latest results. Non-fatal."""
    try:
        folder = report_folder()
        log_path = folder / f"{STATE['fid']}_inspection_log.tsv"
        log = pd.read_csv(log_path, sep='\t') if log_path.exists() else pd.DataFrame(columns=LOG_COLUMNS)
        status_rows = ''.join(f'<tr><td style="padding:2px 10px;">{k} {n}</td><td>{badge_html(k)}</td></tr>'
                              for k, n in SECTIONS)
        blocks = ''
        for k, n in SECTIONS + [('explorer', 'Epoch explorer')]:
            if REPORT_BLOCKS.get(k):
                title = n if k == 'explorer' else f'{k}. {n}'
                blocks += f'<h2>{title}</h2>' + ''.join(REPORT_BLOCKS[k])
        html = ('<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
                f"<title>Inspection {STATE['fid']}</title>"
                '<style>body{font-family:Arial,sans-serif;max-width:1300px;margin:auto;padding:10px;}'
                'td,th{padding:2px 8px;border-bottom:1px solid #eee;text-align:left;}</style></head><body>'
                f"<h1>Inspection of {STATE['fid']}</h1>"
                f"<p>Recording: <code>{STATE['edf']}</code><br>Study folder: <code>{STATE['study']}</code><br>"
                f"Report written {datetime.now().strftime('%Y-%m-%d %H:%M')} by tool 0.</p>"
                f"{STATE.get('registry_html', '')}"
                f'<h2>Steps</h2><table>{status_rows}</table>'
                f'<h2>What was done (all sessions)</h2>{df_html(log)}{blocks}</body></html>')
        out = folder / f"{STATE['fid']}_inspection_report.html"
        out.write_text(html, encoding='utf-8')
        html_report_info.value = f'<small>Inspection report: {out}</small>'
    except Exception as e:
        html_report_info.value = f'<span style="color:#c62828;">⚠ inspection report not written: {e}</span>'


# Colour family of the scored events on the navigator (matched by substring, case-insensitive).
EVENT_COLORS = [('apnea', '#c0392b'), ('hypopnea', '#e67e22'), ('arousal', '#f1c40f'),
                ('desat', '#2980b9'), ('limb', '#27ae60'), ('plm', '#27ae60'), ('movement', '#27ae60')]


def event_color(name):
    low = str(name).lower()
    for key, col in EVENT_COLORS:
        if key in low:
            return col
    return '#8e44ad'

## Study folder and participant

In [ ]:
fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = ('<b>Study folder</b>: the folder holding <code>config_param/</code>, <code>derivatives/</code> '
                 'and the <code>reports_*</code> folders (the recordings are found in any sub-folder)')
dd_part = widgets.Dropdown(description='Participant:', options=[], value=None,
                           style={'description_width': '90px'}, layout=widgets.Layout(width='420px'))
cb_report = widgets.Checkbox(value=True, description='Keep an inspection report (reports_live_explore/)',
                             indent=False, layout=widgets.Layout(width='420px'))
html_study = widgets.HTML('')
html_part = widgets.HTML('')
html_report_info = widgets.HTML('')
html_guard = widgets.HTML('')          # unsaved-rescoring warning

STUDY = {}          # study-wide scan: recordings, configuration files, companion suffixes (tool 7 rules)
NONE = '(none)'


def _in_output_tree(path, study):
    # derivatives/ and reports_* hold outputs, never recordings
    try:
        first = Path(path).relative_to(study).parts[0]
    except Exception:
        return False
    return first == 'derivatives' or first.startswith('reports_')


def scan_study(study):
    """Recordings + configuration files of the study + the companion suffixes tool 7 would pick."""
    study = Path(study)
    edfs, dup = {}, []
    for f in EIO.list_edf_files(study):
        if _in_output_tree(f, study):
            continue
        if f.stem in edfs:
            dup.append(f.stem)
            continue
        edfs[f.stem] = f
    files = [f for f in study.rglob('*') if f.suffix.lower() in ('.txt', '.csv')
             and not _in_output_tree(f, study)]
    txt = [f for f in files if f.suffix.lower() == '.txt']
    csv = [f for f in files if f.suffix.lower() == '.csv']
    edf_list = list(edfs.values())
    hyp_counts = EIO.companion_suffix_counts(edf_list, txt)
    hyp_suffix, _ = EIO.select_hypno_suffix(hyp_counts)
    txt_suffix, _ = EIO.select_event_suffix(
        EIO.companion_suffix_counts(edf_list, [f for f in txt if 'event' in f.name.lower()]))
    csv_suffix, _ = EIO.select_event_suffix(EIO.companion_suffix_counts(edf_list, csv))
    cfg_dir = study / 'config_param'
    config, config_msg = None, "no <code>config_param/remap_reref_persubject.json</code> (tool 2): channels auto-detected"
    if (cfg_dir / 'remap_reref_persubject.json').exists():
        try:
            config = EIO.load_json_lenient(cfg_dir / 'remap_reref_persubject.json')
            config_msg = f"channel configuration of tool 2 read ({len(config)} participant(s))"
        except Exception as e:
            config_msg = f'<span style="color:#c62828;">⚠ tool 2 configuration unreadable: {e}</span>'
    event_remap, event_msg = {}, 'no <code>config_param/event_remap.json</code> (tool 4): raw event labels shown'
    if (cfg_dir / 'event_remap.json').exists():
        try:
            event_remap = EIO.load_event_remap(cfg_dir / 'event_remap.json')
            event_msg = f'event labels of tool 4 read ({len(event_remap)} label(s))'
        except Exception as e:
            event_msg = f'<span style="color:#c62828;">⚠ tool 4 event remap unreadable: {e}</span>'
    STUDY.update(study=study, edfs=edfs, dup=dup, hypno_suffix=hyp_suffix,
                 txt_suffix=txt_suffix or '_ScoredEvents_Export.txt', csv_suffix=csv_suffix or '_event_xml.csv',
                 config=config, config_msg=config_msg, event_remap=event_remap, event_msg=event_msg,
                 custom_stages=L.load_custom_stages(study))


def _on_study(chooser=None):
    try:
        STUDY.clear()
        html_study.value = ''
        dd_part.unobserve(_on_participant, names='value')
        dd_part.options = []
        dd_part.value = None
        dd_part.observe(_on_participant, names='value')
        reset_participant()
        if not fc_data.selected_path:
            return
        scan_study(fc_data.selected_path)
        n = len(STUDY['edfs'])
        dup = (f"<br>⚠ same file name in two sub-folders, only the first kept: {', '.join(STUDY['dup'])}"
               if STUDY['dup'] else '')
        html_study.value = (f"<small>{n} recording(s) found. {STUDY['config_msg']}; {STUDY['event_msg']}. "
                            f"Hypnogram suffix tool 7 would pick: <code>{STUDY['hypno_suffix']}</code>.{dup}</small>")
        dd_part.unobserve(_on_participant, names='value')
        dd_part.options = sorted(STUDY['edfs'])
        dd_part.value = None
        dd_part.observe(_on_participant, names='value')
    except Exception as e:
        html_study.value = f'<span style="color:#c62828;">Study folder error: {e}</span>'


def _on_participant(change):
    try:
        if not guard_unsaved('change the participant'):
            dd_part.unobserve(_on_participant, names='value')
            dd_part.value = change['old']          # stay on the participant whose rescoring is unsaved
            dd_part.observe(_on_participant, names='value')
            return
        reset_participant()
        if change['new'] is None:
            return
        prefill_participant(change['new'])
    except Exception as e:
        html_part.value = f'<span style="color:#c62828;">Could not read this recording: {e}</span>'


fc_data.register_callback(_on_study)
dd_part.observe(_on_participant, names='value')

## Steps

In [ ]:
# ===== Step 1: Load & configuration =====
btn_run1 = widgets.Button(description='Load the recording', icon='play', button_style='primary',
                          layout=widgets.Layout(width='220px', height='36px'))
html_cfg1 = widgets.HTML('')          # what was found for this participant (header only, no signal)
box_eeg1 = widgets.VBox([])           # EEG checkboxes, only when tool 2's configuration has no entry
eeg_cbs = {}
CTX_SLOTS = [('eog_left', 'EOG left'), ('eog_right', 'EOG right'), ('emg', 'EMG'), ('ecg', 'ECG')]
dd_ctx = {role: widgets.Dropdown(description=label, options=[NONE], value=NONE,
                                 style={'description_width': '70px'}, layout=widgets.Layout(width='250px'))
          for role, label in CTX_SLOTS}
html_ctx1 = widgets.HTML('')
dd_hypno = widgets.Dropdown(description='Hypnogram:', options=[NONE], value=NONE,
                            style={'description_width': '90px'}, layout=widgets.Layout(width='480px'))
html_hypno1 = widgets.HTML('')
txt_custom = widgets.Text(description='Custom stages:',
                          placeholder='non-AASM labels kept, e.g. N4 (from config_param/custom_stages.json)',
                          style={'description_width': '110px'}, layout=widgets.Layout(width='480px'))
out_sec1 = widgets.Output()


def registry_status_html(study, fid):
    reg, warn = PSL.load_registry(study)
    excluded, reason = PSL.status_of(PSL.effective_status(reg), fid)
    if warn:
        return f'<p style="color:#e65100;">⚠ {warn}</p>'
    if excluded:
        return (f'<p style="background:#ffebee;color:#b71c1c;padding:4px 8px;"><b>EXCLUDED</b> in the '
                f'exclusion registry: {reason}</p>')
    return ('<p style="color:#1b5e20;">Included (no exclusion in <code>config_param/participant_exclusions.tsv'
            '</code>).</p>')


def prefill_participant(fid):
    """Header-only look at the chosen recording (the signal is read by 'Load the recording'): channels and
    their detected types, tool 2's configuration entry, context channels, hypnograms, event files."""
    edf = STUDY['edfs'][fid]
    STATE.update(study=STUDY['study'], edf=edf, fid=fid)
    hdr = EIO.read_edf_header_custom(str(edf), fallback_encoding='latin-1')
    chans, trans = hdr['channel'], hdr['transducer_type']
    types = EIO.detect_channel_types(chans, trans)
    cfg = EIO.participant_config(STUDY['config'], fid) if STUDY.get('config') else None
    STATE.update(header=hdr, types=types, cfg=cfg)
    STATE['registry_html'] = registry_status_html(STUDY['study'], fid)
    html_part.value = f'<small>{edf}</small>' + STATE['registry_html']

    parts = []
    # EEG channels: tool 2's remap when this participant has an entry, else detection (editable)
    eeg_cbs.clear()
    if cfg is not None and cfg.get('remap'):
        remap = cfg['remap']
        rows = ''.join(f'<tr><td>{o}</td><td>&rarr; {n}</td><td>{"" if o in chans else "⚠ not in the file"}</td></tr>'
                       for o, n in remap.items())
        parts.append(f"<b>EEG channels</b> from tool 2's configuration (re-reference for step 6: "
                     f"<code>{cfg.get('ref_channels')}</code>):<table>{rows}</table>")
        box_eeg1.children = ()
    else:
        for ch in chans:
            eeg_cbs[ch] = widgets.Checkbox(value=(types[ch] == 'EEG'), description=ch, indent=False,
                                           layout=widgets.Layout(width='140px'))
        box_eeg1.children = (
            widgets.HTML("<b>EEG channels</b>: this participant has no entry in tool 2's configuration, so "
                         "they are auto-detected. Tick / untick (names kept as in the file):"),
            widgets.Box(list(eeg_cbs.values()), layout=widgets.Layout(flex_flow='row wrap')))
    # context channels: declared in tool 2's configuration, else detected (editable, never saved)
    declared = (cfg or {}).get('context_channels') or {}
    if declared:
        guess, src = declared, "declared in tool 2's configuration"
    else:
        guess = EIO.detect_context_channels(pd.DataFrame({'channel': chans, 'transducer_type': trans}))
        src = ("auto-detected (no <code>context_channels</code> in tool 2's configuration): check them, "
               'tool 7 only uses declared ones')
    for role, dd in dd_ctx.items():
        dd.options = [NONE] + list(chans)
        dd.value = guess.get(role) if guess.get(role) in chans else NONE
    html_ctx1.value = f'<small>EOG / EMG / ECG context channels {src}.</small>'
    # hypnograms: every .txt named after the recording; tool 7's choice pre-selected
    cands = sorted(f.name[len(edf.stem):] for f in edf.parent.iterdir()
                   if f.suffix.lower() == '.txt' and os.path.normcase(f.name).startswith(os.path.normcase(edf.stem)))
    hyp = [s for s in cands if 'event' not in s.lower()] + [s for s in cands if 'event' in s.lower()]
    dd_hypno.options = [NONE] + hyp
    best = STUDY.get('hypno_suffix')
    dd_hypno.value = best if best in hyp else (hyp[0] if hyp and 'event' not in hyp[0].lower() else NONE)
    html_hypno1.value = (f"<small>Suffix tool 7 would use for the whole study: <code>{best}</code>. "
                         'Pick another one to look at a different scoring (e.g. tool 3bis\'s automatic one).</small>'
                         if best else '<small style="color:#e65100;">No hypnogram found beside the recordings.</small>')
    # events: which companion the TXT -> CSV -> XML chain will read
    txt, csv, xml = EIO.event_companion_paths(edf, STUDY['txt_suffix'], STUDY['csv_suffix'])
    found = [f'{k} <code>{Path(p).name}</code>' for k, p in (('TXT', txt), ('CSV', csv), ('XML', xml)) if p]
    parts.append('<b>Scored events</b>: ' + (', '.join(found) + ' (read in this order of priority, like tools 5 and 7)'
                                             if found else 'no event file beside the recording'))
    sf_eeg = sorted({sf for ch, sf in zip(chans, hdr['sampling_frequency']) if types[ch] == 'EEG'})
    parts.insert(0, f"<b>{edf.name}</b>: {len(chans)} channels in the file, EEG sampled at {', '.join(sf_eeg) or '?'} Hz.")
    html_cfg1.value = '<br>'.join(parts)
    txt_custom.value = ', '.join(STUDY.get('custom_stages', []))
    with out_sec1:
        clear_output()
        print('Check the configuration above, then click "Load the recording".')


def load_recording():
    """Read the signal like tool 7 (EEG as recorded: no re-reference, no filter), the context channels,
    the hypnogram (tool 7's length rule), the scored events and the manual annotations."""
    edf, fid, cfg = Path(STATE['edf']), STATE['fid'], STATE['cfg']
    custom = L.parse_custom_field(txt_custom.value)
    notes = []
    if cfg is not None and cfg.get('remap'):
        raw = EIO.read_eeg_raw(edf, cfg['remap'])
    else:
        sel = [ch for ch, cb in eeg_cbs.items() if cb.value]
        if not sel:
            raise ValueError('no EEG channel selected')
        raw = mne.io.read_raw_edf(str(edf), preload=False, encoding='latin-1', include=sel, verbose=False)
        raw, _ = EIO.drop_suffix_duplicates(raw)
        raw.load_data()
    if not len(raw.ch_names):
        raise ValueError('none of the EEG channels is in the file')
    sf = float(raw.info['sfreq'])
    n_sig = int(np.floor(raw.n_times / sf / 30))
    # hypnogram, with tool 7's length rule
    hpath, status = None, 'none'
    if dd_hypno.value != NONE:
        hpath = edf.with_name(f'{edf.stem}{dd_hypno.value}')
        labels = EIO.load_hypno_labels(hpath)
        n_hyp = len(labels)
        status = EIO.hypno_length_status(n_hyp, n_sig)
        if status == 'trim':
            labels = labels[:n_sig]
            notes.append(f'The recording ends inside its last 30 s epoch: the last score is dropped '
                         f'({n_hyp} &rarr; {n_sig} epochs), as tool 7 does.')
        elif status == 'mismatch':
            notes.append(f'<span style="color:#c62828;"><b>⚠ Length mismatch</b>: the hypnogram has {n_hyp} epochs, '
                         f'the signal {n_sig} complete 30 s epochs. <b>Tool 7 skips this participant.</b> '
                         'Here the hypnogram is cut / padded with "?" so the night can still be looked at.</span>')
            labels = (labels[:n_sig] if n_hyp > n_sig else
                      np.concatenate([labels, np.array(['?'] * (n_sig - n_hyp), dtype='<U10')]))
    else:
        labels = np.array(['?'] * n_sig, dtype='<U10')
        notes.append('No hypnogram: every epoch is "?" (you can score the night in the explorer).')
    unknown = sorted(set(labels) - set(L.AASM_STAGES) - {'MT', '?'} - set(custom))
    if unknown:
        n_unk = int(np.isin(labels, unknown).sum())
        notes.append(f'<span style="color:#e65100;">⚠ {n_unk} epoch(s) carry a label outside W/N1/N2/N3/R: '
                     f'{", ".join(unknown)}. Harmonize them with tool 3, or declare them as custom stages.</span>')
    # scored events (TXT -> CSV -> XML), labels translated with tool 4's event_remap.json
    ev, ev_src = EIO.load_events(edf, STUDY['txt_suffix'], STUDY['csv_suffix'])
    onsets = None
    if ev is not None and len(ev):
        remap_ev = STUDY.get('event_remap') or {}
        canon = [remap_ev.get(n) for n in ev['Name']]
        onsets = pd.DataFrame({'type': [c if isinstance(c, str) and c else str(n) for c, n in zip(canon, ev['Name'])],
                               'raw_label': ev['Name'].astype(str).values,
                               'mapped': [isinstance(c, str) and bool(c) for c in canon],
                               'onset_s': pd.to_numeric(ev['Start'], errors='coerce').values,
                               'duration_s': pd.to_numeric(ev['Duration'], errors='coerce').values})
    manual = L.load_manual_events(edf.parent, fid)
    # context channels (EOG / EMG / ECG), read and display-filtered like tool 7's companion
    ctx_map = {role: dd.value for role, dd in dd_ctx.items() if dd.value != NONE}
    ctx = None
    if ctx_map:
        ctx_raw = EIO.read_context_raw(edf, ctx_map, sf)
        if ctx_raw is not None:
            cep = mne.make_fixed_length_epochs(ctx_raw, duration=30, preload=True, verbose=False)
            ctx = {'data_uV': cep.get_data() * 1e6, 'sfreq': float(cep.info['sfreq']), 'labels': list(cep.ch_names)}
    STATE.update(raw=raw, sf=sf, n_sig=n_sig, hypno_path=hpath, hypno_status=status,
                 hypno_orig=labels.copy(), hypno_work=labels.copy(), hypno_saved=labels.copy(),
                 unknown_labels=unknown, discard_armed=None,
                 events=ev, events_source=ev_src, onsets=onsets, manual_events=manual,
                 ctx=ctx, ctx_map=ctx_map, notes=notes, custom_stages=custom)
    build_display('raw')


def build_display(mode):
    """Epochs of the displayed signal ('raw' = as recorded, 'bp' = display band-pass 0.3-35 Hz), cut like
    tool 7 (30 s from t = 0, as many as the hypnogram), their PSD (detail panel) and the navigator
    spectrogram. A band-pass is for the eyes only: it never reaches a computation of steps 2-8."""
    raw = STATE['raw']
    if mode == 'bp':
        raw = raw.copy().filter(0.3, min(35.0, STATE['sf'] / 2 - 0.5), verbose=False)
    epochs = mne.make_fixed_length_epochs(raw, duration=30, preload=False, verbose=False)
    epochs.drop_bad(verbose=False)            # epochs over a BAD annotation are dropped, as in tool 7
    n = min(len(epochs), len(STATE['hypno_work']))
    if len(epochs) < STATE['n_sig']:
        STATE['notes'].append(f'⚠ MNE dropped {STATE["n_sig"] - len(epochs)} epoch(s) overlapping a BAD '
                              'annotation of the EDF: the epoch numbers after them are shifted.')
    epochs = epochs[:n]
    epochs.metadata = pd.DataFrame({'stage': STATE['hypno_work'][:n], 'reject_flag': np.zeros(n, dtype=bool),
                                    'reject_method': ['none'] * n})
    P = L.participant_from_epochs(epochs)
    freqs, psds = L.compute_psds(P['epochs'], fmax=45.0)
    STATE.update(P=P, freqs=freqs, psds=psds, display_mode=mode, n=n)
    compute_nav_spectrogram()


def sec1_summary():
    """Summary of what was loaded (shown in the tab and kept for the report)."""
    P, labels = STATE['P'], STATE['hypno_work']
    parts = [f"<b>Recording</b>: {STATE['n']} epochs of 30 s ({STATE['n'] * 30 / 3600:.1f} h) at "
             f"{STATE['sf']:.0f} Hz; EEG: {', '.join(P['ch_names'])}; context: "
             f"{', '.join(STATE['ctx']['labels']) if STATE['ctx'] else 'none'}."]
    hname = STATE['hypno_path'].name if STATE['hypno_path'] else 'none'
    counts = pd.Series(labels).value_counts()
    order = [s for s in L.AASM_STAGES + STATE['custom_stages'] + ['MT', '?'] if s in counts.index]
    order += [s for s in counts.index if s not in order]
    stage_tab = pd.DataFrame({'stage': order, 'epochs': [int(counts[s]) for s in order],
                              'minutes': [counts[s] * 0.5 for s in order]})
    parts.append(f'<b>Hypnogram</b>: <code>{hname}</code>' + df_html(stage_tab))
    on = STATE['onsets']
    if on is not None and len(on):
        by = on.groupby('type').agg(n=('onset_s', 'size'), mapped=('mapped', 'first')).reset_index()
        by = by.sort_values('n', ascending=False)
        by['label'] = np.where(by['mapped'], 'canonical (tool 4)', 'raw label, not mapped')
        n_unmapped = int((~on['mapped']).sum())
        warn = (f' <span style="color:#e65100;">{n_unmapped} event(s) have a label tool 4 does not map: '
                'tool 7 ignores them.</span>' if n_unmapped else '')
        parts.append(f"<b>Scored events</b>: {len(on)} from the {STATE['events_source'].upper()} companion.{warn}"
                     + df_html(by[['type', 'n', 'label']]))
    else:
        parts.append('<b>Scored events</b>: none.')
    nm = len(STATE['manual_events'])
    parts.append(f'<b>Manual annotations</b>: {nm} (tool 8 / tool 0, <code>{STATE["fid"]}_manual_events.tsv</code>).')
    if STATE['notes']:
        parts.append('<b>Notes</b><ul>' + ''.join(f'<li>{n}</li>' for n in STATE['notes']) + '</ul>')
    return '<br>'.join(parts)


def on_run1(b):
    b.disabled = True
    try:
        with out_sec1:
            clear_output()
            if 'edf' not in STATE:
                print('Choose a study folder and a participant first.')
                return
            if not guard_unsaved('click "Load the recording"'):
                print('Not reloaded: see the warning at the top.')
                return
            print('Reading the recording…')
        load_recording()
        html_guard.value = ''
        summary = sec1_summary()
        with out_sec1:
            clear_output()
            display(HTML(summary))
        REPORT_BLOCKS['1'] = [summary]
        mark_up_to_date('1')
        explorer_open()
        log_action('1', 'recording loaded',
                   f"hypnogram={STATE['hypno_path'].name if STATE['hypno_path'] else 'none'} ({STATE['hypno_status']}); "
                   f"events={STATE['events_source']}; context={','.join(STATE['ctx_map'].values()) or 'none'}")
    except Exception as e:
        with out_sec1:
            clear_output()
            print(f'Loading failed: {e!r}')
    finally:
        b.disabled = False


btn_run1.on_click(on_run1)
BADGES['1'] = widgets.HTML(badge_html('1'))
sec1_box = widgets.VBox([
    training_box(
        "Reads the recording and gathers its configuration: which channels are EEG (tool 2's remap), which "
        'ones are the EOG / EMG / ECG context, which hypnogram and which scored events belong to it.',
        'Tool 2 declares the channels, tool 3 harmonizes the hypnogram labels and tool 4 the event labels, for '
        'every recording of the study.',
        'Is every EEG channel found? Are the context channels on the right role? Is the hypnogram as long as '
        'the recording (tool 7 skips a participant whose hypnogram does not match)? Are the events read from '
        'the expected file, and are their labels mapped by tool 4?'),
    BADGES['1'], html_cfg1, box_eeg1,
    widgets.HTML('<b>Context channels</b> (shown under the EEG in the explorer):'),
    widgets.HBox(list(dd_ctx.values())), html_ctx1,
    dd_hypno, html_hypno1, txt_custom, btn_run1, out_sec1])

In [ ]:
# ===== Step 2: Recording header =====
btn_run2 = widgets.Button(description='Show the header', icon='play', button_style='primary',
                          layout=widgets.Layout(width='220px', height='36px'))
out_sec2 = widgets.Output()


def channel_roles():
    """Role of each channel of the file in this inspection (EEG analysed / context / not used)."""
    roles = {}
    cfg = STATE.get('cfg')
    if cfg is not None and cfg.get('remap'):
        for o, n in cfg['remap'].items():
            roles[o] = f'EEG (as {n})' if n != o else 'EEG'
    else:
        for ch, cb in eeg_cbs.items():
            if cb.value:
                roles[ch] = 'EEG'
    for role, dd in dd_ctx.items():
        if dd.value != NONE:
            roles[dd.value] = 'context ' + dict(CTX_SLOTS)[role]
    return roles


def sec2_content():
    hdr, edf = STATE['header'], Path(STATE['edf'])
    df = EIO.header_channel_table(hdr)
    roles = channel_roles()
    df.insert(1, 'role here', df['channel'].map(lambda c: roles.get(c, '')))
    df.insert(2, 'detected type', df['channel'].map(STATE['types']))
    df['dynamic_range_uV'] = (df['res_theoretical'] * pow(2, 16)).round(1)
    is_eeg = df['role here'].str.startswith('EEG')
    flags = []
    narrow = df[is_eeg & (df['dynamic_range_uV'] <= EIO.DYNAMIC_RANGE_THRESHOLD_UV)]
    for _, r in narrow.iterrows():
        flags.append(f"EEG <b>{r['channel']}</b>: dynamic range {r['dynamic_range_uV']:.0f} µV "
                     f'(≤ {EIO.DYNAMIC_RANGE_THRESHOLD_UV:.0f} µV): large slow waves and artefacts are clipped.')
    rates = sorted(set(df.loc[is_eeg, 'sampling_frequency']))
    if len(rates) > 1:
        flags.append(f"EEG channels at different rates ({', '.join(rates)} Hz): MNE reads them all at the highest.")
    filt = df.loc[is_eeg, ['highpass', 'lowpass', 'notch']].astype(str).drop_duplicates()
    if len(filt) > 1:
        flags.append('The EEG channels were not exported with the same hardware filters (see the table).')
    anon = EIO.check_anonymization(hdr['patient_id'], edf.stem)
    if anon['header_anonymized']:
        anon_txt = '<span style="color:#1b5e20;">header anonymized</span>'
    else:
        anon_txt = (f"<span style=\"color:#c62828;\"><b>{anon['anon_warning']}</b>: anonymize the recording "
                    'with tool 1bis before sharing it</span>')
        flags.append('The header holds a patient name (not shown here): run tool 1bis.')
    dur_s = hdr['n_data_records'] * hdr['duration_data_record']
    glob = pd.DataFrame({'field': ['EDF version', 'start date and time', 'duration', 'data records',
                                   'channels', 'patient field format'],
                         'value': [hdr['version'],
                                   f"{hdr['start_date']} {hdr['start_time']}",
                                   f'{dur_s / 3600:.2f} h ({dur_s:.0f} s)',
                                   f"{hdr['n_data_records']} × {hdr['duration_data_record']} s",
                                   str(hdr['n_channels']), anon['patient_id_format']]})
    cols = ['channel', 'role here', 'detected type', 'transducer_type', 'dimension', 'sampling_frequency',
            'physical_min', 'physical_max', 'dynamic_range_uV', 'res_theoretical', 'highpass', 'lowpass', 'notch']
    df[['highpass', 'lowpass', 'notch']] = df[['highpass', 'lowpass', 'notch']].astype(object).where(
        df[['highpass', 'lowpass', 'notch']].notna(), '')
    flag_html = ('<ul>' + ''.join(f'<li>{f}</li>' for f in flags) + '</ul>' if flags
                 else '<p style="color:#1b5e20;">Nothing to report.</p>')
    return (f'<p>Anonymization: {anon_txt}</p>' + df_html(glob)
            + f'<h4>Checks</h4>{flag_html}<h4>Channels</h4>' + df_html(df[cols]))


def on_run2(b):
    b.disabled = True
    try:
        with out_sec2:
            clear_output()
            if 'header' not in STATE:
                print('Choose a study folder and a participant first.')
                return
        content = sec2_content()
        with out_sec2:
            clear_output()
            display(HTML(content))
        REPORT_BLOCKS['2'] = [content]
        mark_up_to_date('2')
        log_action('2', 'header shown')
    except Exception as e:
        with out_sec2:
            clear_output()
            print(f'Header error: {e!r}')
    finally:
        b.disabled = False


btn_run2.on_click(on_run2)
BADGES['2'] = widgets.HTML(badge_html('2'))
sec2_box = widgets.VBox([
    training_box(
        'Shows how the recording was exported: per channel, the sampling rate, unit, physical range and '
        'hardware filters written in the EDF header (the signal is not read).',
        'Tool 1 inspects the headers of the whole database (one report per dataset and per participant); '
        'tool 1bis anonymizes them.',
        'Is the header anonymized? Are all EEG channels at the same rate? An exported dynamic range of 500 µV or '
        'less clips large slow waves and artefacts. Are the hardware filters the expected ones?'),
    BADGES['2'], btn_run2, out_sec2])

In [ ]:
# ===== Steps 3 to 8: built in the next phases of the tool-0 redesign =====
PLANNED = {
    '3': ('Shows the hypnogram and the scored events with their labels in the toolkit vocabulary; optionally '
          'scores the night automatically (GSSC) and compares it with the expert scoring.',
          'Tool 3 harmonizes the hypnogram labels, tool 4 the event labels, tool 3bis scores automatically.',
          'Labels left unmapped, long stretches where the automatic and expert scorings disagree, events outside '
          'the night.', 2),
    '4': ('Computes the classical PSG summary: time in bed, total sleep time, sleep efficiency, latencies, '
          'WASO, stage durations, apnea-hypopnea and limb-movement indices.',
          'Tool 5 computes it for every recording; tool 5bis selects the participants on these values.',
          'Values outside the reference ranges, and where lights-off / lights-on come from.', 3),
    '5': ('Measures the quality of each EEG channel over the whole night: flat signal, clipping at the EDF '
          'bounds, amplitude distribution, spectrogram.',
          'Tool 6 does it for every recording and channel.',
          'A flagged channel, a two-peak amplitude histogram (drift), a dead or saturated stretch in the '
          'spectrogram.', 4),
    '6': ('Preprocesses the EEG like tool 7 (re-reference, notch, band-pass) and flags each 30 s epoch and '
          'channel (amplitude, gradient, flat signal, 1/f fit, scored events).',
          'Tool 7 does it for every recording and writes the epochs used by tools 8, 8bis and 9.',
          'Which methods flag, on which channels, in which sleep stages.', 5),
    '7': ('Turns the flags into a keep / reject decision: drop the channels flagged too often, then reject the '
          'epochs flagged on too many of the remaining channels.',
          'Tool 8bis decides automatically; tool 8 lets a reviewer decide epoch by epoch.',
          'Dropped channels, share of rejected epochs per stage, would this participant be excluded?', 6),
    '8': ('Spectral features on the kept epochs: power per band and stage, aperiodic (1/f) exponent and offset, '
          'oscillatory peaks.',
          'Tool 9 computes them for every recording and builds the database tables.',
          'Spectra per stage that look physiological (alpha in wake, spindles in N2, slow waves in N3).', 7),
}
planned_boxes = {}
for _k, (_what, _batch, _look, _phase) in PLANNED.items():
    BADGES[_k] = widgets.HTML(badge_html(_k))
    planned_boxes[_k] = widgets.VBox([
        training_box(_what, _batch, _look), BADGES[_k],
        widgets.HTML(f'<p style="color:#757575;"><i>This step is not built yet (tool-0 redesign, phase {_phase}). '
                     'Until then, run the batch tool named above.</i></p>')])

## Epoch explorer

In [ ]:
# ===== Epoch explorer: the night epoch by epoch, with tool 8's display =====
# Fixed clinical scales (µV per channel row) from the shared library: an amplitude keeps the same height from
# one epoch to the next, which is what makes visual criteria (75 µV slow waves, EMG tone...) usable.
_sclay, _scsty = widgets.Layout(width='120px'), {'description_width': '40px'}
ft_sc = {t: widgets.FloatText(description=t, value=L.DISPLAY_SCALE_UV[t], step=step, style=_scsty, layout=_sclay)
         for t, step in (('EEG', 25), ('EOG', 50), ('EMG', 25), ('ECG', 100))}
dd_signal = widgets.Dropdown(description='Signal:', options=[('as recorded', 'raw'),
                                                            ('display band-pass 0.3-35 Hz', 'bp')],
                             value='raw', style={'description_width': '55px'}, layout=widgets.Layout(width='300px'))
dd_nav_ch = widgets.Dropdown(description='Spectrogram:', options=[], style={'description_width': '90px'},
                             layout=widgets.Layout(width='230px'))
dd_channels = widgets.Dropdown(description='Channels:', value='all',
                               options=[('all', 'all'), ('flagged only', 'flagged'),
                                        ('flagged + neighbours', 'flagged+neighbours'), ('worst 16 (p-p)', 'worst')],
                               style={'description_width': '70px'}, layout=widgets.Layout(width='250px'))
sl_ctx = widgets.IntSlider(description='± context', value=1, min=0, max=3, layout=widgets.Layout(width='230px'))
cb_context = widgets.Checkbox(value=True, description='Show EOG/EMG/ECG', indent=False,
                              layout=widgets.Layout(width='170px'))
ft_epoch_offset = widgets.IntText(description='Epoch n° offset', value=1, style={'description_width': '110px'},
                                  layout=widgets.Layout(width='180px'))
sl_epoch = widgets.IntSlider(description='Epoch', value=0, min=0, max=0, continuous_update=False,
                             layout=widgets.Layout(width='520px'))
btn_prev = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
btn_next = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))
btn_next_stage = widgets.Button(description='Next stage change', layout=widgets.Layout(width='150px'))
lbl_epoch = widgets.HTML('')
SCORE_STAGES = ['W', 'N1', 'N2', 'N3', 'R', 'MT', '?']
stage_btns = {s: widgets.Button(description=s, button_style='warning', layout=widgets.Layout(width='52px'))
              for s in SCORE_STAGES}
cb_autoadv = widgets.Checkbox(value=True, description='Auto-advance after scoring', indent=False,
                              layout=widgets.Layout(width='210px'))
btn_save_hypno = widgets.Button(description='Save the rescored hypnogram', icon='save', button_style='success',
                                layout=widgets.Layout(width='250px'))
cb_overwrite = widgets.Checkbox(value=False, description='Overwrite an existing _rescored.txt', indent=False,
                                layout=widgets.Layout(width='260px'))
out_save = widgets.Output()
dd_annot_label = widgets.Dropdown(description='Label:', options=L.MANUAL_EVENT_LABELS,
                                  style={'description_width': '45px'}, layout=widgets.Layout(width='240px'))
ft_annot_offset = widgets.FloatText(description='at (s) in epoch', value=0.0, step=1.0,
                                    style={'description_width': '105px'}, layout=widgets.Layout(width='190px'))
txt_annot_comment = widgets.Text(description='Comment:', placeholder='free text: the event name when the label is "other"',
                                 style={'description_width': '70px'}, layout=widgets.Layout(width='380px'))
btn_annot_add = widgets.Button(description='Add annotation', icon='plus', layout=widgets.Layout(width='160px'))
box_annot_list = widgets.VBox([])
lbl_annot_info = widgets.HTML('')
out_nav = widgets.Output()
out_montage = widgets.Output()
out_detail = widgets.Output()
html_explorer_info = widgets.HTML('<i>Load a recording (step 1) to open the explorer.</i>')
KEY_TO_STAGE = {'w': 'W', '0': 'W', '1': 'N1', '2': 'N2', '3': 'N3', '4': 'R', '5': 'R', 'r': 'R', 'm': 'MT'}


def compute_nav_spectrogram():
    """Navigator spectrogram of the chosen channel: 30 s windows (50 % overlap) over the displayed signal,
    so its time axis is in epochs and a cursor move only redraws, never recomputes."""
    P = STATE['P']
    ch = dd_nav_ch.value if dd_nav_ch.value in P['ch_names'] else P['ch_names'][0]
    sig = P['data_uV'][:, P['ch_names'].index(ch), :].reshape(-1)
    nper = max(64, int(P['sfreq'] * 30))
    f, t, sxx = scipy_spectrogram(sig, fs=P['sfreq'], nperseg=nper, noverlap=nper // 2)
    keep = f <= 30
    sxx = sxx[keep]
    db = 10 * np.log10(np.where(sxx > 0, sxx, np.nan))       # µV²/Hz, log guarded (no epsilon floor)
    STATE['nav_spec'] = (t / 30.0, f[keep], db)


def plot_navigator(cur):
    """Spectrogram of the night with the hypnogram on top (white line), scored events (ticks at the bottom),
    manual annotations (violet), rescored epochs (orange marks) and the current epoch (red cursor)."""
    n = STATE['n']
    labels = STATE['hypno_work'][:n]
    fig, ax = plt.subplots(figsize=(13, 2.8))
    t_ep, f, db = STATE['nav_spec']
    finite = db[np.isfinite(db)]
    if finite.size:
        # p5-p99: the silent floor and artefact spikes would otherwise squash the useful ~40 dB band
        vmin, vmax = float(np.percentile(finite, 5)), float(np.percentile(finite, 99))
        ax.pcolormesh(t_ep, f, db, cmap='viridis', vmin=vmin, vmax=vmax, shading='auto', rasterized=True)
    ax.set_ylim(0.5, 30)
    ax.set_ylabel('Freq (Hz)', fontsize=8)
    ax.set_xlim(0, n)
    ax.set_xlabel('Epoch index', fontsize=8)
    ax.tick_params(labelsize=7)
    ax2 = ax.twinx()
    stage_y, _col, ytp, ytl = L.custom_stage_style(STATE['custom_stages'])
    y = [stage_y.get(s, min(ytp) - 1) for s in labels]
    ax2.step(np.arange(n + 1), y + [y[-1]], where='post', color='white', linewidth=1.7, zorder=6,
             path_effects=[mpe.Stroke(linewidth=3.4, foreground='black'), mpe.Normal()])
    ax2.set_yticks(ytp)
    ax2.set_yticklabels(ytl, fontsize=8)
    ax2.set_ylim(min(ytp) - 1.3, max(ytp) + 1.0)
    ax2.set_xlim(0, n)
    for ei in np.where(np.asarray(labels) != np.asarray(STATE['hypno_orig'][:n]))[0]:
        ax2.plot([ei + 0.5], [max(ytp) + 0.7], marker='v', color='#e67e22', markersize=7, zorder=8)
    on = STATE.get('onsets')
    if on is not None and len(on):
        for typ, grp in on.groupby('type'):
            ax2.plot(grp['onset_s'].values / 30.0, np.full(len(grp), min(ytp) - 0.9), linestyle='none',
                     marker='|', color=event_color(typ), markersize=10, markeredgewidth=2, zorder=7)
    man = STATE.get('manual_events')
    if man is not None and len(man):
        ax2.plot(pd.to_numeric(man['onset_s'], errors='coerce').values / 30.0, np.full(len(man), min(ytp) - 0.5),
                 linestyle='none', marker='v', color=L.MANUAL_COLOR, markersize=7, zorder=8)
    ax2.axvline(cur + 0.5, color='#ff1744', linewidth=2.5, zorder=10)
    plt.tight_layout()
    return fig


def preview_thresholds():
    # Until step 6 runs, the trace colours preview tool 7's time-domain criteria at its default thresholds
    return {k: (dict(v) if isinstance(v, dict) else v) for k, v in L.DEFAULT_THRESHOLDS.items()}


def epoch_identity(ei):
    num, elapsed, clock = L.epoch_labels(ei, 30, STATE['P'].get('meas_date'), int(ft_epoch_offset.value))
    return num, elapsed, clock


def clock_at(onset_s):
    md = STATE['P'].get('meas_date')
    if md is None:
        return ''
    try:
        return (md + pd.to_timedelta(float(onset_s), unit='s')).strftime('%H:%M:%S')
    except Exception:
        return ''


def status_html(ei):
    num, elapsed, clock = epoch_identity(ei)
    stage, orig = STATE['hypno_work'][ei], STATE['hypno_orig'][ei]
    changed = f' <span style="color:#e65100;">(rescored, was {orig})</span>' if stage != orig else ''
    n_mod = int((STATE['hypno_work'] != STATE['hypno_orig']).sum())
    clock_txt = f' · clock {clock}' if clock else ''
    n_unsaved = unsaved_rescoring()
    unsaved = (f' <span style="color:#e65100;">({n_unsaved} not saved)</span>' if n_unsaved else '')
    return (f'<b>Epoch {ei} (#{num})</b> · {elapsed} since the start{clock_txt} · stage <b>{stage}</b>{changed}'
            f' · {n_mod} epoch(s) rescored{unsaved}')


def render_explorer():
    if 'P' not in STATE:
        return
    ei = STATE['cur']
    thr = preview_thresholds()
    P = STATE['P']
    lbl_epoch.value = status_html(ei)
    with out_nav:
        clear_output(wait=True)
        try:
            display(widgets.Image(value=fig_to_png(plot_navigator(ei)), format='png'))
        except Exception as e:
            print(f'Navigator error: {e}')
    with out_montage:
        clear_output(wait=True)
        try:
            num, _elapsed, clock = epoch_identity(ei)
            scored, manual = L.merge_onsets_for_display(STATE.get('onsets'), STATE.get('manual_events'))
            show = None if dd_channels.value == 'all' else L.select_montage_channels(P, ei, thr, mode=dd_channels.value)
            fig = L.plot_epoch_montage(P, ei, thr, context=int(sl_ctx.value),
                                       ctx=STATE['ctx'] if cb_context.value else None,
                                       onsets=scored, title_method='step 6 not run',
                                       scales={t: w.value for t, w in ft_sc.items()}, show_channels=show,
                                       number_label=f'(#{num}' + (f', {clock}' if clock else '') + ')',
                                       manual=manual)
            display(widgets.Image(value=fig_to_png(fig), format='png'))
        except Exception as e:
            print(f'Montage error: {e}')
    with out_detail:
        clear_output(wait=True)
        try:
            fig = L.plot_epoch_detail(P, ei, STATE['freqs'], STATE['psds'], thr, fmin=2.0)
            display(widgets.Image(value=fig_to_png(fig), format='png'))
        except Exception as e:
            print(f'Detail error: {e}')
    refresh_annot_list(ei)


def goto_epoch(i):
    if 'P' not in STATE:
        return
    i = int(np.clip(i, 0, STATE['n'] - 1))
    STATE['cur'] = i
    if sl_epoch.value != i:
        sl_epoch.value = i                    # its observer redraws
    else:
        render_explorer()


def _on_slider(change):
    if 'P' in STATE:
        STATE['cur'] = int(change['new'])
        render_explorer()


def _next_stage_change(b):
    if 'P' not in STATE:
        return
    labels, cur = STATE['hypno_work'], STATE['cur']
    nxt = next((j for j in range(cur + 1, STATE['n']) if labels[j] != labels[cur]), STATE['n'] - 1)
    goto_epoch(nxt)


def score_current(stage):
    """Assign a stage to the current epoch (stage buttons and keyboard). Held in memory until saved; the
    steps computed from the hypnogram become out of date."""
    if 'P' not in STATE:
        return
    ei = STATE['cur']
    if STATE['hypno_work'][ei] != stage:
        STATE['hypno_work'][ei] = stage
        STATE['P']['stages'][ei] = stage
        STATE['P']['meta'].at[ei, 'stage'] = stage
        mark_out_of_date(AFTER_RESCORING, 'the hypnogram was rescored')
    if cb_autoadv.value and ei < STATE['n'] - 1:
        goto_epoch(ei + 1)
    else:
        render_explorer()


def on_save_hypno(b):
    with out_save:
        clear_output()
        try:
            if 'P' not in STATE:
                print('Load a recording first.')
                return
            base = STATE['hypno_path'] if STATE['hypno_path'] is not None else Path(STATE['edf'])
            out_txt = base.with_name(f'{base.stem}_rescored.txt')
            out_log = base.with_name(f'{base.stem}_rescore_log.tsv')
            if out_txt.exists() and not cb_overwrite.value:
                print(f'⚠ {out_txt.name} already exists: tick "Overwrite an existing _rescored.txt" to replace it.')
                return
            work, orig = STATE['hypno_work'], STATE['hypno_orig']
            np.savetxt(str(out_txt), work, fmt='%s')
            STATE['hypno_saved'] = work.copy()
            STATE['discard_armed'] = None
            html_guard.value = ''
            now = datetime.now().isoformat(timespec='seconds')
            rows = [{'epoch_idx': int(ei), 'epoch_time_sec': int(ei * 30), 'original_label': str(orig[ei]),
                     'new_label': str(work[ei]), 'timestamp': now} for ei in np.where(work != orig)[0]]
            pd.DataFrame(rows, columns=['epoch_idx', 'epoch_time_sec', 'original_label', 'new_label',
                                        'timestamp']).to_csv(str(out_log), sep='\t', index=False)
            print(f'Saved {out_txt.name} ({len(work)} epochs, {len(rows)} rescored) and {out_log.name} '
                  f'in {out_txt.parent}. The original hypnogram is untouched.')
            REPORT_BLOCKS.setdefault('explorer', [])
            REPORT_BLOCKS['explorer'] = [b_ for b_ in REPORT_BLOCKS['explorer'] if 'id="rescoring"' not in b_]
            REPORT_BLOCKS['explorer'].append(f'<p id="rescoring">Rescoring saved to <code>{out_txt}</code>: '
                                             f'{len(rows)} epoch(s) changed.</p>')
            log_action('explorer', 'rescored hypnogram saved', f'{out_txt.name}: {len(rows)} epoch(s) changed')
        except Exception as e:
            print(f'Save failed: {e!r}')


# ----- Manual annotations: the same file and format as tool 8, beside the recording -----
def annot_label_options():
    opts = [v for v in (STUDY.get('event_remap') or {}).values() if isinstance(v, str) and v]
    opts += L.MANUAL_EVENT_LABELS
    return list(dict.fromkeys(opts))


def annots_of_epoch(ei):
    df = STATE.get('manual_events')
    if df is None or not len(df):
        return []
    return list(df.index[pd.to_numeric(df['epoch_idx'], errors='coerce') == int(ei)])


def save_annots():
    p = L.save_manual_events(Path(STATE['edf']).parent, STATE['fid'], STATE['manual_events'])
    if p is None:
        lbl_annot_info.value = '<span style="color:#c62828;">⚠ could not write the annotation file.</span>'
    else:
        lbl_annot_info.value = f"<small>{len(STATE['manual_events'])} annotation(s) &rarr; {p}</small>"
        mark_out_of_date(AFTER_ANNOTATION, 'an annotation was added or deleted')
        man = STATE['manual_events']
        REPORT_BLOCKS.setdefault('explorer', [])
        REPORT_BLOCKS['explorer'] = [b_ for b_ in REPORT_BLOCKS['explorer'] if 'id="annotations"' not in b_]
        REPORT_BLOCKS['explorer'].append(f'<div id="annotations"><p>Manual annotations ({len(man)}), in '
                                         f'<code>{p}</code>:</p>'
                                         + df_html(man[['type', 'onset_s', 'epoch_number', 'clock_time', 'stage',
                                                        'comment', 'source']]) + '</div>')


def _make_annot_delete(row_id):
    def _f(b):
        try:
            row = STATE['manual_events'].loc[row_id]
            STATE['manual_events'] = STATE['manual_events'].drop(index=row_id)
            save_annots()
            log_action('explorer', 'annotation deleted', f"{row['type']} at {float(row['onset_s']):.1f} s")
            render_explorer()
        except Exception as e:
            lbl_annot_info.value = f'<span style="color:#c62828;">Delete error: {e}</span>'
    return _f


def refresh_annot_list(ei):
    try:
        rows = []
        for rid in annots_of_epoch(ei):
            a = STATE['manual_events'].loc[rid]
            cmt = str(a.get('comment', '') or '')
            cmt = '' if cmt.lower() == 'nan' else cmt
            txt = (f'<b>{a["type"]}</b> @ {float(a["onset_s"]):.1f} s {a.get("clock_time", "")} '
                   f'<small>({a.get("source", "")})</small>' + (f': {cmt}' if cmt else ''))
            btn = widgets.Button(description='', icon='trash', tooltip='Delete this annotation',
                                 layout=widgets.Layout(width='40px'))
            btn.on_click(_make_annot_delete(rid))
            rows.append(widgets.HBox([btn, widgets.HTML(txt)]))
        box_annot_list.children = tuple(rows)
    except Exception as e:
        box_annot_list.children = (widgets.HTML(f'<span style="color:#c62828;">Annotation list error: {e}</span>'),)


def on_annot_add(b):
    # Written to disk at once: a Voila session that dies must never take an annotation with it.
    try:
        if 'P' not in STATE:
            lbl_annot_info.value = '<span style="color:#c62828;">Load a recording first.</span>'
            return
        ei = STATE['cur']
        label = str(dd_annot_label.value)
        comment = txt_annot_comment.value.strip()
        if label == 'other':
            if not comment:
                lbl_annot_info.value = ('<span style="color:#e65100;">Type the event name in Comment when the '
                                        'label is "other".</span>')
                return
            label = comment
        off = min(max(float(ft_annot_offset.value), 0.0), 30.0)    # the mark stays inside its own epoch
        onset = ei * 30.0 + off
        num, _elapsed, _clock = epoch_identity(ei)
        row = {'type': label, 'onset_s': round(onset, 3), 'duration_s': np.nan, 'epoch_idx': int(ei),
               'epoch_number': num, 'clock_time': clock_at(onset), 'stage': STATE['hypno_work'][ei],
               'comment': comment, 'source': 'manual_tool0',
               'created_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
        new = pd.DataFrame([row])
        STATE['manual_events'] = (new if not len(STATE['manual_events'])
                                  else pd.concat([STATE['manual_events'], new], ignore_index=True))
        save_annots()
        txt_annot_comment.value = ''
        log_action('explorer', 'annotation added', f'{label} at {onset:.1f} s (epoch {ei})')
        render_explorer()
    except Exception as e:
        lbl_annot_info.value = f'<span style="color:#c62828;">Annotation error: {e}</span>'


def explorer_open():
    """Called once step 1 has loaded a recording: sync the widgets and draw the first epoch."""
    P = STATE['P']
    dd_nav_ch.unobserve(_on_nav_ch, names='value')
    dd_nav_ch.options = P['ch_names']
    dd_nav_ch.value = P['ch_names'][0]
    dd_nav_ch.observe(_on_nav_ch, names='value')
    dd_signal.unobserve(_on_signal, names='value')
    dd_signal.value = STATE['display_mode']
    dd_signal.observe(_on_signal, names='value')
    dd_annot_label.options = annot_label_options()
    sl_epoch.unobserve(_on_slider, names='value')
    sl_epoch.max = max(0, STATE['n'] - 1)
    sl_epoch.value = 0
    sl_epoch.observe(_on_slider, names='value')
    STATE['cur'] = 0
    html_explorer_info.value = (
        '<small>Fixed clinical scales (µV per row), so amplitudes compare from one epoch to the next. Trace '
        "colours preview tool 7's time-domain criteria at its <b>default</b> thresholds on the displayed signal: "
        'step 6 computes the real flags. PSD background: median and IQR of the epochs of the same stage. '
        + ('Keyboard (click the navigator or the montage first): ←/→ epochs, w/0 = W, 1, 2, 3, r/4/5 = R, m = MT.'
           if _HAS_IPYEVENTS else 'Install ipyevents for keyboard shortcuts.') + '</small>')
    lbl_annot_info.value = (f"<small>{len(STATE['manual_events'])} annotation(s) in "
                            f"<code>{STATE['fid']}_manual_events.tsv</code> beside the recording.</small>")
    render_explorer()


def explorer_reset():
    for o in (out_nav, out_montage, out_detail, out_save):
        o.clear_output()
    lbl_epoch.value = ''
    box_annot_list.children = ()
    lbl_annot_info.value = ''
    html_explorer_info.value = '<i>Load a recording (step 1) to open the explorer.</i>'


def _on_nav_ch(change):
    if 'P' in STATE:
        compute_nav_spectrogram()
        render_explorer()


def _on_signal(change):
    if 'P' not in STATE:
        return
    try:
        lbl_epoch.value = '<i>Computing the displayed signal…</i>'
        cur = STATE['cur']
        build_display(change['new'])
        STATE['cur'] = min(cur, STATE['n'] - 1)
        render_explorer()
    except Exception as e:
        lbl_epoch.value = f'<span style="color:#c62828;">Display error: {e}</span>'


btn_prev.on_click(lambda b: goto_epoch(STATE.get('cur', 0) - 1))
btn_next.on_click(lambda b: goto_epoch(STATE.get('cur', 0) + 1))
btn_next_stage.on_click(_next_stage_change)
for _s, _b in stage_btns.items():
    _b.on_click(lambda b, st=_s: score_current(st))
btn_save_hypno.on_click(on_save_hypno)
btn_annot_add.on_click(on_annot_add)
sl_epoch.observe(_on_slider, names='value')
dd_nav_ch.observe(_on_nav_ch, names='value')
dd_signal.observe(_on_signal, names='value')
for _w in list(ft_sc.values()) + [dd_channels, sl_ctx, cb_context, ft_epoch_offset]:
    _w.observe(lambda c: render_explorer(), names='value')

figures_box = widgets.VBox([out_nav, out_montage])      # the keyboard listens here (no text field inside)
explorer_box = widgets.VBox([
    html_explorer_info,
    widgets.HBox([dd_signal, dd_nav_ch, dd_channels]),
    widgets.HBox([sl_ctx, cb_context, ft_epoch_offset]),
    widgets.HBox([widgets.HTML('<b>Scale (µV/row):</b>')] + list(ft_sc.values())),
    widgets.HBox([btn_prev, sl_epoch, btn_next, btn_next_stage]),
    lbl_epoch,
    figures_box,
    widgets.HBox([widgets.HTML('<b>Rescore:</b>')] + list(stage_btns.values()) + [cb_autoadv]),
    widgets.HBox([widgets.HTML('<b>Annotate:</b>'), dd_annot_label, ft_annot_offset, txt_annot_comment,
                  btn_annot_add]),
    box_annot_list, lbl_annot_info,
    out_detail,
    widgets.HBox([btn_save_hypno, cb_overwrite]), out_save])

if _HAS_IPYEVENTS:
    _kb = _IpyEvent(source=figures_box, watched_events=['keydown'], prevent_default_action=True)

    def _on_key(event):
        k = event.get('key', '')
        if k == 'ArrowRight':
            goto_epoch(STATE.get('cur', 0) + 1)
        elif k == 'ArrowLeft':
            goto_epoch(STATE.get('cur', 0) - 1)
        elif k.lower() in KEY_TO_STAGE:
            score_current(KEY_TO_STAGE[k.lower()])

    _kb.on_dom_event(_on_key)

In [ ]:
# ===== Page layout =====
def reset_participant():
    """A new participant (or study): every step and the explorer start from scratch."""
    reset_state()
    for o in (out_sec1, out_sec2):
        o.clear_output()
    html_part.value = ''
    html_cfg1.value = ''
    html_ctx1.value = ''
    html_hypno1.value = ''
    html_report_info.value = ''
    html_guard.value = ''
    box_eeg1.children = ()
    explorer_reset()


TABS = widgets.Tab(children=[sec1_box, sec2_box] + [planned_boxes[k] for k in ('3', '4', '5', '6', '7', '8')])
refresh_badges()
explorer_acc = widgets.Accordion(children=[explorer_box])
explorer_acc.set_title(0, 'Epoch explorer')
explorer_acc.selected_index = 0
display(fc_data, html_study, widgets.HBox([dd_part, cb_report]), html_part, html_guard, html_report_info,
        HTML('<h3>Steps</h3>'), TABS, HTML('<h3>Epoch explorer</h3>'), explorer_acc)